<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 37 · PPO và Gymnasium</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Cài PPO với GAE và môi trường song song của Gymnasium, huấn luyện trên CartPole, trên Pendulum với hành động liên tục, và trên một môi trường quản lý kho do chính bạn viết. Chương cuối Phần III-B; mỗi lần huấn luyện 20 tới 60 giây trên CPU.</p>
</div>

### Sau notebook này bạn sẽ

- Thấy bằng autograd phép cắt của PPO chặn gradient ở đâu.
- Cài PPO: môi trường vector, tự động reset, GAE, nhiều epoch trên mỗi lô.
- Đổi sang chính sách chuẩn để điều khiển hành động liên tục.
- Viết một môi trường Gymnasium, kiểm bằng check_env, và so PPO với chính sách (s, S) của vận trù học.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> khoảng 7 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch37-ppo-gymnasium/01-buoc-an-toan/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Bước an toàn](#trang-1)
2. [Trang 2 · PPO trên CartPole](#trang-2)
3. [Trang 3 · Hành động liên tục](#trang-3)
4. [Trang 4 · Tự viết môi trường](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Bước an toàn

Mục tiêu cắt của PPO, kiểm bằng autograd.

*Đọc kèm trên web: [Trang 1 · Bước an toàn](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch37-ppo-gymnasium/01-buoc-an-toan/)*

In [ ]:
!pip install -q gymnasium                                   # thư viện môi trường RL, Colab chưa chắc có sẵn

In [ ]:
import gymnasium as gym                                     # môi trường RL
import numpy as np                                          # mảng
import torch                                                # PyTorch
import torch.nn as nn                                       # các lớp mạng
torch.set_num_threads(1)                                    # mạng nhỏ: một luồng, kết quả lặp lại được
ti_so = torch.tensor([0.5, 0.9, 1.0, 1.1, 1.3, 1.5], requires_grad=True)   # sáu giá trị của rho
for A in (1.0, -1.0):                                       # lợi thế dương và âm
    L = torch.min(ti_so * A, ti_so.clamp(0.8, 1.2) * A)     # mục tiêu cắt, epsilon = 0,2
    (g,) = torch.autograd.grad(L.sum(), ti_so)              # gradient theo rho
    print(A, L.detach().numpy().round(2), g.numpy())        # giá trị và gradient

Output mong đợi:

```text
1.0 [0.5 0.9 1.  1.1 1.2 1.2] [1. 1. 1. 1. 0. 0.]
-1.0 [-0.8 -0.9 -1.  -1.1 -1.3 -1.5] [ 0. -1. -1. -1. -1. -1.]
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Ngoài khoảng 0,8 tới 1,2 về phía có lợi, gradient bằng 0: không đẩy tiếp. Về phía bất lợi, gradient vẫn khác 0: sai lầm luôn bị kéo về.</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · PPO trên CartPole

Cài PPO và huấn luyện trên CartPole (khoảng 22 giây).

*Đọc kèm trên web: [Trang 2 · PPO trên CartPole](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch37-ppo-gymnasium/02-ppo-tren-cartpole/)*

In [ ]:
def lop(vao, ra, std=np.sqrt(2)):                           # lớp tuyến tính khởi tạo trực giao
    l = nn.Linear(vao, ra)                                  # lớp thường
    nn.init.orthogonal_(l.weight, std)                      # trọng số trực giao
    nn.init.zeros_(l.bias)                                  # bias 0
    return l                                                # trả về lớp
class TacTu(nn.Module):                                     # diễn viên và nhà phê bình
    def __init__(self, n_obs, n_ra, lien_tuc=False):        # số chiều quan sát, số hành động hoặc số chiều hành động
        super().__init__()                                  # khởi tạo nn.Module
        self.lien_tuc = lien_tuc                            # hành động liên tục hay rời rạc
        self.critic = nn.Sequential(lop(n_obs, 64), nn.Tanh(), lop(64, 64), nn.Tanh(), lop(64, 1, 1.0))   # nhà phê bình V(s)
        self.actor = nn.Sequential(lop(n_obs, 64), nn.Tanh(), lop(64, 64), nn.Tanh(), lop(64, n_ra, 0.01))   # diễn viên, lớp cuối rất nhỏ
        self.log_std = nn.Parameter(torch.zeros(n_ra)) if lien_tuc else None   # độ lệch chuẩn học được (trang 3)
    def phan_phoi(self, o):                                 # phân phối hành động
        if self.lien_tuc:                                   # liên tục: phân phối chuẩn
            mu = self.actor(o)                              # trung bình
            return torch.distributions.Normal(mu, self.log_std.exp().expand_as(mu))   # chuẩn với sigma chung
        return torch.distributions.Categorical(logits=self.actor(o))   # rời rạc: softmax của logit
    def log_p_ent(self, o, a):                              # log xác suất và entropy
        d = self.phan_phoi(o)                               # phân phối hiện tại
        if self.lien_tuc:                                   # nhiều chiều độc lập
            return d.log_prob(a).sum(-1), d.entropy().sum(-1)   # cộng qua các chiều
        return d.log_prob(a), d.entropy()                   # một hành động rời rạc
    def tham_lam(self, o):                                  # dùng khi đánh giá
        return self.actor(o) if self.lien_tuc else self.actor(o).argmax(-1)   # trung bình hoặc argmax
def gae(R, V, V_sau, ket_thuc, het_tap, gamma, lam):        # lợi thế GAE, đi ngược thời gian
    A = torch.zeros_like(R)                                 # kết quả
    g = torch.zeros(R.shape[1])                             # tích lũy cho từng môi trường
    for t in reversed(range(R.shape[0])):                   # từ cuối lô về đầu
        delta = R[t] + gamma * (1 - ket_thuc[t]) * V_sau[t] - V[t]   # sai số TD; chỉ terminated mới bỏ V_sau
        g = delta + gamma * lam * (1 - het_tap[t]) * g      # không tích lũy qua ranh giới tập
        A[t] = g                                            # ghi lại
    return A                                                # lợi thế
def ppo(tao_mot, tong_buoc, n_env=4, n_buoc=128, epoch=4, n_lo=4, lr=2.5e-4, gamma=0.99, lam=0.95,
        clip=0.2, he_so_ent=0.01, lien_tuc=False, seed=0, in_moi=40):   # PPO đầy đủ
    torch.manual_seed(seed)                                 # khởi tạo và lấy mẫu
    envs = gym.vector.SyncVectorEnv([lambda: gym.wrappers.RecordEpisodeStatistics(tao_mot()) for _ in range(n_env)],
                                    autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)   # n_env bản sao, tự reset
    n_obs = envs.single_observation_space.shape[0]          # số chiều quan sát
    n_ra = envs.single_action_space.shape[0] if lien_tuc else envs.single_action_space.n   # số đầu ra của diễn viên
    tt = TacTu(n_obs, n_ra, lien_tuc)                       # hai mạng
    opt = torch.optim.Adam(tt.parameters(), lr=lr, eps=1e-5, fused=True)   # fused: lặp lại được trên CPU
    so_vong = tong_buoc // (n_env * n_buoc)                 # số lô
    o, _ = envs.reset(seed=seed)                            # quan sát đầu của mọi bản sao
    o = torch.tensor(o, dtype=torch.float32)                # sang tensor
    diem = []                                               # điểm các tập đã xong
    for vong in range(1, so_vong + 1):                      # mỗi lô
        opt.param_groups[0]["lr"] = lr * (1 - (vong - 1) / so_vong)   # giảm learning rate tuyến tính
        O, A, LP, V, R, V_sau, KT, HT = [], [], [], [], [], [], [], []   # dữ liệu của lô
        for t in range(n_buoc):                             # 1. thu một lô bằng chính sách hiện tại
            with torch.no_grad():                           # thu dữ liệu, không cần gradient
                d = tt.phan_phoi(o)                         # phân phối hành động
                a = d.sample()                              # lấy mẫu
                lp = d.log_prob(a).sum(-1) if lien_tuc else d.log_prob(a)   # log xác suất CŨ, giữ cố định
                v = tt.critic(o).squeeze(-1)                # V(s)
            o2, r, ket_thuc, cat_ngang, info = envs.step(a.numpy())   # n_env bước cùng lúc
            het = ket_thuc | cat_ngang                      # bản sao nào vừa hết tập
            o_cuoi = o2.copy()                              # trạng thái thật sự đến
            for i in np.where(het)[0]:                      # với bản sao vừa hết tập
                o_cuoi[i] = info["final_obs"][i]            # o2 đã là trạng thái của tập mới
                diem.append(float(info["final_info"]["episode"]["r"][i]))   # điểm do RecordEpisodeStatistics ghi
            with torch.no_grad():                           # không cần gradient
                vs = tt.critic(torch.tensor(o_cuoi, dtype=torch.float32)).squeeze(-1)   # V(s_t+1)
            O.append(o); A.append(a); LP.append(lp); V.append(v); V_sau.append(vs)   # lưu
            R.append(torch.tensor(r, dtype=torch.float32))  # phần thưởng
            KT.append(torch.tensor(ket_thuc, dtype=torch.float32))   # kết thúc thật
            HT.append(torch.tensor(het, dtype=torch.float32))   # hết tập (kể cả bị cắt)
            o = torch.tensor(o2, dtype=torch.float32)       # sang bước sau
        O, A, LP, V, R, V_sau, KT, HT = map(torch.stack, (O, A, LP, V, R, V_sau, KT, HT))   # n_buoc x n_env
        with torch.no_grad():                               # 2. lợi thế bằng GAE
            adv = gae(R, V, V_sau, KT, HT, gamma, lam)      # lợi thế
            muc_tieu_v = adv + V                            # mục tiêu cho nhà phê bình
        N = n_buoc * n_env                                  # số mẫu trong lô
        O, A, LP, adv, muc_tieu_v = O.flatten(0, 1), A.flatten(0, 1), LP.flatten(0, 1), adv.flatten(0, 1), muc_tieu_v.flatten(0, 1)   # gộp hai chiều đầu
        co_lo = N // n_lo                                   # cỡ mini-batch
        for _ in range(epoch):                              # 3. học nhiều epoch trên cùng một lô
            thu_tu = torch.randperm(N)                      # xáo trộn
            for k in range(n_lo):                           # mỗi mini-batch
                j = thu_tu[k * co_lo:(k + 1) * co_lo]       # chỉ số của mini-batch
                lp, ent = tt.log_p_ent(O[j], A[j])          # log xác suất MỚI
                ti_so = (lp - LP[j]).exp()                  # rho = pi mới / pi cũ
                a_j = (adv[j] - adv[j].mean()) / (adv[j].std() + 1e-8)   # chuẩn hóa lợi thế
                loss_pi = -torch.min(ti_so * a_j, ti_so.clamp(1 - clip, 1 + clip) * a_j).mean()   # mục tiêu cắt, đổi dấu
                loss_v = ((tt.critic(O[j]).squeeze(-1) - muc_tieu_v[j]) ** 2).mean()   # nhà phê bình
                loss = loss_pi + 0.5 * loss_v - he_so_ent * ent.mean()   # thưởng entropy
                opt.zero_grad()                             # xóa gradient
                loss.backward()                             # lượt ngược
                nn.utils.clip_grad_norm_(tt.parameters(), 0.5)   # cắt độ lớn gradient
                opt.step()                                  # cập nhật
        if vong % in_moi == 0:                              # định kỳ
            print(vong * N, round(np.mean(diem[-20:]), 1))  # số bước, điểm 20 tập gần nhất
    return tt                                               # tác tử đã học
def danh_gia(tt, tao_mot, n=20, seed=10_000):               # chính sách tham lam trên n tập cố định
    e = tao_mot()                                           # môi trường riêng
    tong = []                                               # điểm mỗi tập
    for i in range(n):                                      # n tập
        o, _ = e.reset(seed=seed + i)                       # seed cố định
        G = 0                                               # điểm
        while True:                                         # mỗi bước
            with torch.no_grad():                           # không cần gradient
                a = tt.tham_lam(torch.tensor(o, dtype=torch.float32)).numpy()   # không lấy mẫu
            o, r, ket_thuc, cat_ngang, _ = e.step(a)        # đi
            G += r                                          # cộng thưởng
            if ket_thuc or cat_ngang:                       # hết tập
                break                                       # ra khỏi vòng
        tong.append(G)                                      # ghi lại
    return round(float(np.mean(tong)), 1)                   # trung bình
CP = lambda: gym.make("CartPole-v1")                        # hàm tạo môi trường
tt_cp = ppo(CP, 200_000)                                    # 200 000 bước
print(danh_gia(tt_cp, CP, n=100, seed=20_000))              # 100 tập test chưa dùng

Output mong đợi:

```text
20480 118.6
40960 256.9
61440 222.4
81920 273.8
102400 352.5
122880 312.7
143360 342.4
163840 351.5
184320 341.5
484.7
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Điểm lúc học còn lấy mẫu và còn thưởng entropy; chính sách tham lam trên 100 tập test được 484,7.</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Hành động liên tục

Pendulum: hành động là một số thực (khoảng 21 giây).

*Đọc kèm trên web: [Trang 3 · Hành động liên tục](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch37-ppo-gymnasium/03-hanh-dong-lien-tuc/)*

In [ ]:
PE = lambda: gym.make("Pendulum-v1")                        # con lắc ngược
e = PE()                                                    # một bản để xem
print(e.observation_space, e.action_space, e.spec.max_episode_steps)   # cos, sin, vận tốc góc; mô-men trong [-2, 2]

Output mong đợi:

```text
Box([-1. -1. -8.], [1. 1. 8.], (3,), float32) Box(-2.0, 2.0, (1,), float32) 200
```

In [ ]:
rng = np.random.default_rng(0)                              # ngẫu nhiên
tong = []                                                   # điểm mỗi tập
for i in range(20):                                         # 20 tập
    o, _ = e.reset(seed=10_000 + i)                         # seed cố định
    G = 0                                                   # điểm
    while True:                                             # mỗi bước
        o, r, ket_thuc, cat_ngang, _ = e.step(rng.uniform(-2, 2, size=1).astype(np.float32))   # mô-men ngẫu nhiên
        G += r                                              # cộng thưởng
        if ket_thuc or cat_ngang:                           # hết 200 bước
            break                                           # ra khỏi vòng
    tong.append(G)                                          # ghi lại
print(round(np.mean(tong), 1))                              # mốc chọn bừa

Output mong đợi:

```text
-1112.0
```

In [ ]:
tt_pe = ppo(PE, 100_000, n_buoc=512, epoch=10, n_lo=32, lr=1e-3, gamma=0.9, he_so_ent=0.0, lien_tuc=True, in_moi=5)   # chính sách chuẩn
print(danh_gia(tt_pe, PE, n=100, seed=20_000), round(tt_pe.log_std.exp().item(), 3))   # điểm test, độ lệch chuẩn đã học

Output mong đợi:

```text
10240 -1261.0
20480 -1085.1
30720 -750.9
40960 -207.7
51200 -772.8
61440 -479.7
71680 -190.6
81920 -241.7
92160 -193.5
-184.7 0.389
```

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Tự viết môi trường

Môi trường quản lý kho tự viết, chính sách (s, S), và PPO (khoảng 40 giây).

*Đọc kèm trên web: [Trang 4 · Tự viết môi trường](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch37-ppo-gymnasium/04-tu-viet-moi-truong/)*

In [ ]:
import warnings                                             # tắt cảnh báo của check_env
from gymnasium import spaces                                # không gian quan sát và hành động
from gymnasium.utils.env_checker import check_env           # bộ kiểm môi trường
class Kho(gym.Env):                                         # môi trường quản lý kho
    def __init__(self, so_ngay=60, nhu_cau=3.0, suc_chua=20, phi_luu=0.5, phat_thieu=4.0, phi_dat=3.0, gia_hang=1.0):   # tham số bài toán
        self.so_ngay, self.nhu_cau, self.suc_chua = so_ngay, nhu_cau, suc_chua   # kỳ 60 ngày, Poisson trung bình 3, chứa 20
        self.phi_luu, self.phat_thieu, self.phi_dat, self.gia_hang = phi_luu, phat_thieu, phi_dat, gia_hang   # bốn loại chi phí
        self.observation_space = spaces.Box(0.0, 1.0, (2,), np.float32)   # tồn kho, ngày; cả hai chia về [0, 1]
        self.action_space = spaces.Discrete(11)             # đặt 0 tới 10 thùng
    def _quan_sat(self):                                    # quan sát hiện tại
        return np.array([self.ton / self.suc_chua, self.ngay / self.so_ngay], np.float32)   # chuẩn hóa về [0, 1]
    def reset(self, seed=None, options=None):               # bắt đầu kỳ mới
        super().reset(seed=seed)                            # khởi tạo self.np_random
        self.ton, self.ngay = 5, 0                          # 5 thùng, ngày 0
        return self._quan_sat(), {}                         # quan sát, thông tin
    def step(self, a):                                      # một ngày
        dat = min(int(a), self.suc_chua - self.ton)         # không đặt quá sức chứa
        self.ton += dat                                     # hàng về ngay
        cau = int(self.np_random.poisson(self.nhu_cau))     # khách muốn mua
        ban = min(cau, self.ton)                            # bán được
        thieu = cau - ban                                   # thiếu
        self.ton -= ban                                     # còn lại
        self.ngay += 1                                      # sang ngày sau
        chi_phi = (self.phi_luu * self.ton + self.phat_thieu * thieu
                   + (self.phi_dat if dat > 0 else 0) + self.gia_hang * dat)   # lưu kho, thiếu hàng, phí đặt, tiền hàng
        return self._quan_sat(), -chi_phi, False, self.ngay >= self.so_ngay, {"thieu": thieu}   # hết kỳ là bị cắt, không phải kết thúc
with warnings.catch_warnings():                             # cảnh báo duy nhất: không có chế độ vẽ hình
    warnings.simplefilter("ignore")                         # bỏ qua
    check_env(Kho())                                        # kiểm reset, step, seed, kiểu dữ liệu
print("hợp lệ")                                             # không lỗi

Output mong đợi:

```text
hợp lệ
```

In [ ]:
def chinh_sach_sS(s_, S_, n, seed):                         # tồn kho <= s thì đặt cho đủ S
    e = Kho()                                               # môi trường
    tong = []                                               # tổng phần thưởng mỗi kỳ
    for i in range(n):                                      # n kỳ
        e.reset(seed=seed + i)                              # seed cố định
        G = 0                                               # tổng
        while True:                                         # mỗi ngày
            _, r, ket_thuc, cat_ngang, _ = e.step(min(10, S_ - e.ton) if e.ton <= s_ else 0)   # quy tắc (s, S)
            G += r                                          # cộng
            if ket_thuc or cat_ngang:                       # hết kỳ
                break                                       # ra khỏi vòng
        tong.append(G)                                      # ghi lại
    return round(float(np.mean(tong)), 1)                   # trung bình
luoi = sorted((chinh_sach_sS(s_, S_, 500, 0), s_, S_) for s_ in range(6) for S_ in range(s_ + 1, 13))   # thử mọi cặp
print(luoi[-3:])                                            # ba cặp tốt nhất
_, s_tot, S_tot = luoi[-1]                                  # cặp tốt nhất
print(chinh_sach_sS(s_tot, S_tot, 2000, 20_000))            # đo lại trên 2000 kỳ test

Output mong đợi:

```text
[(-379.2, 1, 8), (-376.6, 2, 8), (-376.1, 2, 7)]
-374.9
```

In [ ]:
tt_kho = ppo(Kho, 300_000, n_buoc=512, epoch=10, n_lo=8, lr=1e-3, in_moi=50)   # cùng hàm ppo
print(danh_gia(tt_kho, Kho, n=2000, seed=20_000))           # cùng 2000 kỳ test
with torch.no_grad():                                       # không cần gradient
    print([int(tt_kho.tham_lam(torch.tensor([t / 20, 0.5]))) for t in range(13)])   # số thùng đặt ở ngày 30 theo tồn kho 0 tới 12

Output mong đợi:

```text
102400 -424.2
204800 -401.2
-377.6
[7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">PPO tốn 377,6 so với 374,9 của (s, S) tốt nhất, và tự học ra đúng dạng (s, S): hết hàng hoặc còn 1 thùng thì đặt 7, còn lại không đặt.</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Đoán trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch37-ppo-gymnasium/05-bai-tap/)*

### Bài 37.1 · Bỏ phép cắt

(a) PPO trên CartPole với clip=100.0. (b) 30 epoch mỗi lô, có cắt và không cắt.
<details><summary>Xem đáp án</summary>

(a) Lên 436 rồi sụp về 134 chỉ một lô sau; test 210,8 so với 484,7. (b) Có cắt: 500 từ bước 40 960 và giữ nguyên; không cắt: nhảy loạn từ 31 tới 500. Phép cắt làm việc dùng lại dữ liệu trở nên an toàn. Code ở hai ô ngay dưới (ô thứ hai khoảng 2 phút).
</details>

In [ ]:
tt = ppo(CP, 200_000, clip=100.0)                           # tỷ số không bao giờ chạm ngưỡng
print(danh_gia(tt, CP, n=100, seed=20_000))                 # điểm test

Output mong đợi:

```text
20480 125.0
40960 225.8
61440 349.4
81920 412.8
102400 436.2
122880 133.7
143360 202.7
163840 214.4
184320 178.5
210.8
```

In [ ]:
for clip in (0.2, 100.0):                                   # có cắt, không cắt
    tt = ppo(CP, 200_000, epoch=30, clip=clip)              # 30 epoch mỗi lô
    print("clip", clip, danh_gia(tt, CP, n=100, seed=20_000))   # điểm test

Output mong đợi:

```text
20480 222.1
40960 500.0
61440 500.0
81920 500.0
102400 500.0
122880 500.0
143360 500.0
163840 500.0
184320 500.0
clip 0.2 500.0
20480 450.4
40960 260.2
61440 31.0
81920 429.2
102400 123.5
122880 225.0
143360 108.6
163840 500.0
184320 500.0
clip 100.0 500.0
```

### Bài 37.2 · Hai đầu của GAE

lambda = 0 và lambda = 1.
<details><summary>Xem đáp án</summary>

lambda = 0 hỏng hẳn (19,7); lambda = 1 học được (498,3) nhưng gồ ghề hơn. Code ở ô ngay dưới.
</details>

In [ ]:
for lam in (0.0, 1.0):                                      # TD một bước, lợi nhuận thật
    tt = ppo(CP, 200_000, lam=lam)                          # chỉ đổi lambda
    print("lambda", lam, danh_gia(tt, CP, n=100, seed=20_000))   # điểm test

Output mong đợi:

```text
20480 64.0
40960 56.6
61440 33.0
81920 30.7
102400 27.8
122880 22.5
143360 22.8
163840 20.8
184320 19.2
lambda 0.0 19.7
20480 89.6
40960 173.6
61440 325.2
81920 325.0
102400 393.4
122880 381.2
143360 291.8
163840 374.2
184320 463.3
lambda 1.0 498.3
```

### Bài 37.3 · Khách khó tính hơn

Phạt thiếu hàng 10 thay vì 4: tìm lại (s, S), huấn luyện lại PPO.
<details><summary>Xem đáp án</summary>

(s, S) tốt nhất thành (3, 9), test 412,8; PPO 415,4 và tự nâng ngưỡng đặt hàng từ 1 lên 3. Code ở ô ngay dưới.
</details>

In [ ]:
Kho10 = lambda: Kho(phat_thieu=10.0)                        # thiếu hàng đắt hơn
def chinh_sach_sS_10(s_, S_, n, seed):                      # (s, S) trên môi trường mới
    e = Kho10()                                             # môi trường
    tong = []                                               # tổng mỗi kỳ
    for i in range(n):                                      # n kỳ
        e.reset(seed=seed + i)                              # seed cố định
        G = 0                                               # tổng
        while True:                                         # mỗi ngày
            _, r, ket_thuc, cat_ngang, _ = e.step(min(10, S_ - e.ton) if e.ton <= s_ else 0)   # quy tắc (s, S)
            G += r                                          # cộng
            if ket_thuc or cat_ngang:                       # hết kỳ
                break                                       # ra khỏi vòng
        tong.append(G)                                      # ghi lại
    return round(float(np.mean(tong)), 1)                   # trung bình
luoi = sorted((chinh_sach_sS_10(s_, S_, 500, 0), s_, S_) for s_ in range(6) for S_ in range(s_ + 1, 13))   # lưới
print(luoi[-3:], chinh_sach_sS_10(luoi[-1][1], luoi[-1][2], 2000, 20_000))   # ba cặp tốt nhất, đo lại cặp tốt nhất
tt10 = ppo(Kho10, 300_000, n_buoc=512, epoch=10, n_lo=8, lr=1e-3, in_moi=50)   # huấn luyện lại
print(danh_gia(tt10, Kho10, n=2000, seed=20_000))           # điểm test
with torch.no_grad():                                       # không cần gradient
    print([int(tt10.tham_lam(torch.tensor([t / 20, 0.5]))) for t in range(13)])   # chính sách mới

Output mong đợi:

```text
[(-416.6, 3, 10), (-415.7, 3, 8), (-415.3, 3, 9)] -412.8
102400 -474.9
204800 -429.9
-415.4
[7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0]
```